In [ ]:
import io
import re
import random
import shutil
import math
import xml.etree.ElementTree as ET
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from PIL import Image
from tqdm.auto import tqdm
from collections import defaultdict

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
pasta_origem = Path(
    '/content/drive/Shareddrives/COMPUTER VISION_TURMA 2026/DADOS/Deteccao_Placas'                                    # pasta do dataset de dados
)

arquivo_xml = pasta_origem / 'annotations.xml'
pasta_imagens = pasta_origem / 'images'

pasta_saida = Path(
    '/content/drive/Shareddrives/COMPUTER VISION_TURMA 2026/PROJETO FINAL/GRUPO C/Projeto — Placas Mercosul/dados'    # pasta onde será salvo as imagens modificadas
)


PROPORCAO_TREINO = 0.70                                                                                               # proporção de dados para treino(é bom analisar se ta ok)
SEMENTE = 42                                                                                                          # semente para que a divisão siga um padrão reproduzivel

random.seed(SEMENTE)


In [ ]:
arvore = ET.parse(arquivo_xml)
raiz = arvore.getroot()

recortes = defaultdict(list)

imagens_descartadas = 0
imagens_sem_anotacao = 0
anotacoes_invalidas = 0

for imagem in tqdm(raiz.findall('.//image'), desc="Lendo XML"):

    nome_imagem = imagem.get('name')
    caminho_imagem = pasta_imagens / nome_imagem

    anotacoes = imagem.findall('points')

    # Descartar imagem inteira se possuir classe estrangeira
    if any(
        anotacao.get('label', '').strip().lower() == 'estrangeira'
        for anotacao in anotacoes
    ):
        imagens_descartadas += 1
        continue

    if not anotacoes:
        imagens_sem_anotacao += 1
        continue

    if not caminho_imagem.exists():
        continue

    for indice, anotacao in enumerate(anotacoes):

        classe = anotacao.get('label').strip()

        # Extrair coordenadas dos pontos
        pontos = anotacao.get('points', '').split(';')

        try:
            coordenadas = [
                tuple(map(float, ponto.split(',')))
                for ponto in pontos
                if ponto.strip()
            ]

            xs = [p[0] for p in coordenadas]
            ys = [p[1] for p in coordenadas]

            # Encontrar limites do retângulo
            x1 = math.floor(min(xs))
            y1 = math.floor(min(ys))
            x2 = math.ceil(max(xs))
            y2 = math.ceil(max(ys))

            if x2 <= x1 or y2 <= y1:
                anotacoes_invalidas += 1
                continue

            nome_recorte = f"{Path(nome_imagem).stem}_{indice}.jpg"

            recortes[classe].append({
                'imagem': caminho_imagem,
                'nome': nome_recorte,
                'coordenadas': (x1, y1, x2, y2)
            })

        except (ValueError, IndexError):
            anotacoes_invalidas += 1
            continue

print("\nLeitura concluída!")

print("Imagens descartadas:", imagens_descartadas)
print("Imagens sem anotação:", imagens_sem_anotacao)
print("Anotações inválidas:", anotacoes_invalidas)

for classe, itens in recortes.items():
    print(f"{classe}: {len(itens)} recortes")

Lendo XML:   0%|          | 0/12000 [00:00<?, ?it/s]


Leitura concluída!
Imagens descartadas: 2
Imagens sem anotação: 55
Anotações inválidas: 2
tradicional: 7319 recortes
tradicional moto: 788 recortes
mercosul moto: 2191 recortes
mercosul: 9907 recortes


In [ ]:
divisoes = {}

for classe, itens in recortes.items():

    random.shuffle(itens)

    quantidade_treino = int(len(itens) * PROPORCAO_TREINO)

    divisoes[classe] = {
        'treino': itens[:quantidade_treino],
        'validacao': itens[quantidade_treino:]
    }

print("Divisão realizada!\n")

for classe, divisao in divisoes.items():

    print(
        f"{classe}: "
        f"{len(divisao['treino'])} treino | "
        f"{len(divisao['validacao'])} validação"
    )

Divisão realizada!

tradicional: 5123 treino | 2196 validação
tradicional moto: 551 treino | 237 validação
mercosul moto: 1533 treino | 658 validação
mercosul: 6934 treino | 2973 validação


In [ ]:
total_salvas = 0

for classe, divisao in divisoes.items():

    for conjunto in ['treino', 'validacao']:

        pasta_destino = pasta_saida / conjunto / classe
        pasta_destino.mkdir(parents=True, exist_ok=True)

        for item in tqdm(
            divisao[conjunto],
            desc=f"{classe} - {conjunto}"
        ):

            with Image.open(item['imagem']) as imagem:

                imagem = imagem.convert('RGB')

                x1, y1, x2, y2 = item['coordenadas']

                # Garantir que as coordenadas estejam dentro da imagem
                x1 = max(0, min(x1, imagem.width))
                y1 = max(0, min(y1, imagem.height))
                x2 = max(0, min(x2, imagem.width))
                y2 = max(0, min(y2, imagem.height))

                if x2 <= x1 or y2 <= y1:
                    continue

                recorte = imagem.crop((x1, y1, x2, y2))

                caminho_salvar = pasta_destino / item['nome']

                recorte.save(caminho_salvar, quality=95)

                total_salvas += 1

print(f"\nProcessamento concluído! {total_salvas} imagens salvas.")

In [ ]:
print("RESUMO DO DATASET\n")

total_treino = 0
total_validacao = 0

for classe in recortes:

    pasta_treino = pasta_saida / 'treino' / classe
    pasta_validacao = pasta_saida / 'validacao' / classe

    quantidade_treino = len(list(pasta_treino.glob('*.jpg')))
    quantidade_validacao = len(list(pasta_validacao.glob('*.jpg')))

    total = quantidade_treino + quantidade_validacao

    total_treino += quantidade_treino
    total_validacao += quantidade_validacao

    print(f"Classe: {classe}")
    print(f"  Treino: {quantidade_treino}")
    print(f"  Validação: {quantidade_validacao}")
    print(f"  Total: {total}\n")

print("---------------------------")
print("Total de treino:", total_treino)
print("Total de validação:", total_validacao)
print("Total geral:", total_treino + total_validacao)
print("---------------------------")
print("\nDataset salvo em:", pasta_saida)